In [1]:
"""
Backpropagation explained step-by-step with PyTorch
---------------------------------------------------
This program trains a simple feedforward neural network on the MNIST dataset.
It illustrates:
- Forward propagation
- Loss computation
- Backward propagation (autograd)
- Weight updates
- Manual gradient verification (optional)
"""

'\nBackpropagation explained step-by-step with PyTorch\n---------------------------------------------------\nThis program trains a simple feedforward neural network on the MNIST dataset.\nIt illustrates:\n- Forward propagation\n- Loss computation\n- Backward propagation (autograd)\n- Weight updates\n- Manual gradient verification (optional)\n'

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
import numpy as np

In [3]:
# ---------------------------
# 1. Load and preprocess data (open source: MNIST)
# ---------------------------
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  # mean and std of MNIST
])

# Load full training set
trainset = torchvision.datasets.MNIST(root='./data', train=True,
                                       download=True, transform=transform)
# Use a subset for faster demonstration (first 2000 samples)
trainset = Subset(trainset, range(2000))
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)

# Test set (full for evaluation)
testset = torchvision.datasets.MNIST(root='./data', train=False,
                                      download=True, transform=transform)
testloader = DataLoader(testset, batch_size=64, shuffle=False)

100%|██████████| 9.91M/9.91M [00:00<00:00, 46.7MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.11MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 10.2MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.36MB/s]


In [4]:
# ---------------------------
# 2. Define the neural network architecture
# ---------------------------
class FeedforwardNN(nn.Module):
    """A simple feedforward network with two hidden layers."""
    def __init__(self):
        super(FeedforwardNN, self).__init__()
        self.fc1 = nn.Linear(28*28, 128)   # input layer -> hidden 1
        self.fc2 = nn.Linear(128, 64)       # hidden 1 -> hidden 2
        self.fc3 = nn.Linear(64, 10)        # hidden 2 -> output (10 classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        # Flatten the image (batch_size, 1, 28, 28) -> (batch_size, 784)
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)                     # no softmax: CrossEntropyLoss includes it
        return x

# Instantiate the network, loss function, and optimizer
net = FeedforwardNN()
criterion = nn.CrossEntropyLoss()            # combines LogSoftmax and NLLLoss
optimizer = optim.SGD(net.parameters(), lr=0.01, momentum=0.9)

In [5]:
# ---------------------------
# 3. Training loop with backpropagation explained
# ---------------------------
print("Starting training...\n")
for epoch in range(3):                        # loop over the dataset 3 times
    running_loss = 0.0
    for i, (inputs, labels) in enumerate(trainloader):
        # ---------- Forward pass ----------
        # Compute predicted outputs by passing inputs to the network
        outputs = net(inputs)                  # (batch_size, 10)
        loss = criterion(outputs, labels)      # scalar tensor

        # ---------- Backward pass (the core of backpropagation) ----------
        # Zero the parameter gradients (otherwise they accumulate)
        optimizer.zero_grad()

        # BACKPROPAGATION STARTS HERE
        # This single line computes the gradient of the loss w.r.t. all parameters
        # using the chain rule (automatic differentiation). Internally, PyTorch:
        #   - traverses the computational graph backwards from loss
        #   - applies the chain rule at each node
        #   - accumulates gradients in each parameter's .grad attribute
        loss.backward()

        # (Optional) Inspect gradients of the first layer's weights
        if i == 0 and epoch == 0:
            print("\nAfter first backward pass:")
            print(f"  Gradient of fc1.weight : mean={net.fc1.weight.grad.mean():.6f}, "
                  f"std={net.fc1.weight.grad.std():.6f}")
            print(f"  Gradient of fc1.bias   : mean={net.fc1.bias.grad.mean():.6f}, "
                  f"std={net.fc1.bias.grad.std():.6f}")

        # ---------- Weight update ----------
        # Update parameters using the computed gradients (SGD step)
        optimizer.step()

        # Statistics
        running_loss += loss.item()
        if i % 100 == 99:                       # print every 100 mini-batches
            print(f"Epoch {epoch+1}, batch {i+1:3d}  loss: {running_loss/100:.4f}")
            running_loss = 0.0

print("\nFinished training.\n")

Starting training...


After first backward pass:
  Gradient of fc1.weight : mean=0.000019, std=0.002583
  Gradient of fc1.bias   : mean=0.000257, std=0.002689

Finished training.



In [6]:
# ---------------------------
# 4. Evaluate the network on test data
# ---------------------------
correct = 0
total = 0
with torch.no_grad():                           # disable gradient tracking
    for inputs, labels in testloader:
        outputs = net(inputs)
        _, predicted = torch.max(outputs, 1)    # get the class with highest score
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Test accuracy: {100 * correct / total:.2f}%")

# ---------------------------
# 5. BONUS: Manual backpropagation for one batch (to demystify autograd)
# ---------------------------
print("\n" + "="*60)
print("MANUAL BACKPROPAGATION VERIFICATION (for one batch)")
print("="*60)

# Take a single batch
data_iter = iter(trainloader)
images, labels = next(data_iter)

# ---- Forward pass manually ----
# We'll use the same network but detach to avoid interfering with the trained model
x = images.view(images.size(0), -1).detach().requires_grad_(True)

# Manually compute each layer (keeping intermediate values)
z1 = x @ net.fc1.weight.T + net.fc1.bias        # linear 1
a1 = torch.relu(z1)                              # activation 1
z2 = a1 @ net.fc2.weight.T + net.fc2.bias        # linear 2
a2 = torch.relu(z2)                              # activation 2
z3 = a2 @ net.fc3.weight.T + net.fc3.bias        # linear 3 (logits)

# Loss: cross-entropy (simplified: we'll use PyTorch's function)
loss_manual = criterion(z3, labels)

# ---- Backward pass manually (gradient of loss w.r.t. fc1.weight) ----
# Let's compute dL/dW1 using the chain rule:
# dL/dW1 = (dL/dz3) * (dz3/da2) * (da2/dz2) * (dz2/da1) * (da1/dz1) * (dz1/dW1)

# First, get the gradient of loss w.r.t. z3 (output logits) manually:
# For cross-entropy with softmax, dL/dz3 = softmax(z3) - y_onehot
softmax = torch.exp(z3) / torch.exp(z3).sum(dim=1, keepdim=True)
y_onehot = torch.eye(10)[labels]                 # convert labels to one-hot
dL_dz3 = (softmax - y_onehot) / z3.shape[0]      # divide by batch size (as in criterion)

# Backprop through fc3: dL/dW3 = a2.T @ dL/dz3, dL/db3 = sum(dL/dz3, dim=0)
dL_da2 = dL_dz3 @ net.fc3.weight                  # gradient w.r.t a2

# Through ReLU of second layer: dL/dz2 = dL/da2 * 1{z2>0}
dL_dz2 = dL_da2.clone()
dL_dz2[z2 <= 0] = 0                               # ReLU derivative

# Backprop through fc2: dL/dW2 = a1.T @ dL/dz2
dL_da1 = dL_dz2 @ net.fc2.weight

# Through ReLU of first layer: dL/dz1 = dL/da1 * 1{z1>0}
dL_dz1 = dL_da1.clone()
dL_dz1[z1 <= 0] = 0

# Finally, dL/dW1 = x.T @ dL/dz1
dL_dW1_manual = x.T @ dL_dz1

# Compare with PyTorch's autograd computed gradient for the same batch
# We need to hook into the existing network's gradients. Since we already trained,
# we'll temporarily zero gradients and run a forward/backward on this batch.
net.zero_grad()
outputs = net(images)                             # forward using the trained net
loss = criterion(outputs, labels)
loss.backward()                                   # autograd computes gradients
dL_dW1_auto = net.fc1.weight.grad.clone()

# Compare
print("\nGradient of fc1.weight (first 5x5 corner):")
print("  Autograd  :\n", dL_dW1_auto[:5, :5])
print("  Manual    :\n", dL_dW1_manual[:5, :5])
print("\nMax absolute difference:",
      torch.max(torch.abs(dL_dW1_auto - dL_dW1_manual)).item())

# Show that they are essentially identical (up to floating point)
if torch.allclose(dL_dW1_auto, dL_dW1_manual, rtol=1e-5):
    print("✓ Manual gradients match PyTorch autograd! Backpropagation works.")
else:
    print("✗ Discrepancy – check your manual derivation.")

Test accuracy: 85.18%

MANUAL BACKPROPAGATION VERIFICATION (for one batch)

Gradient of fc1.weight (first 5x5 corner):
  Autograd  :
 tensor([[-0.0020, -0.0020, -0.0020, -0.0020, -0.0020],
        [-0.0005, -0.0005, -0.0005, -0.0005, -0.0005],
        [ 0.0027,  0.0027,  0.0027,  0.0027,  0.0027],
        [ 0.0023,  0.0023,  0.0023,  0.0023,  0.0023],
        [ 0.0010,  0.0010,  0.0010,  0.0010,  0.0010]])
  Manual    :
 tensor([[-0.0020, -0.0005,  0.0027,  0.0023,  0.0010],
        [-0.0020, -0.0005,  0.0027,  0.0023,  0.0010],
        [-0.0020, -0.0005,  0.0027,  0.0023,  0.0010],
        [-0.0020, -0.0005,  0.0027,  0.0023,  0.0010],
        [-0.0020, -0.0005,  0.0027,  0.0023,  0.0010]],
       grad_fn=<SliceBackward0>)


RuntimeError: The size of tensor a (784) must match the size of tensor b (128) at non-singleton dimension 1

In [ ]:
##CIFAR

In [7]:
import os, time, math
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import DataLoader

In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# 0.  REPRODUCIBILITY
# ─────────────────────────────────────────────────────────────────────────────
torch.manual_seed(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Using device: {DEVICE}")

[INFO] Using device: cpu


In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# 1.  DATA PIPELINE  (CIFAR-10, 32×32 RGB, 10 classes)
# ─────────────────────────────────────────────────────────────────────────────
CLASSES = ("plane","car","bird","cat","deer","dog","frog","horse","ship","truck")
BATCH   = 256

train_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    # CIFAR-10 channel-wise mean/std
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010)),
])
test_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010)),
])

train_ds = torchvision.datasets.CIFAR10(root="./data", train=True,
                                        download=True, transform=train_tf)
test_ds  = torchvision.datasets.CIFAR10(root="./data", train=False,
                                        download=True, transform=test_tf)

train_ld = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2)
test_ld  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=2)

100%|██████████| 170M/170M [00:01<00:00, 105MB/s]


In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# 2.  MODEL  – Deep Feedforward Network (MLP) with advanced regularisation
#
#   Input : 32×32×3 = 3072 raw pixels (flattened)
#   Hidden: 4 residual-style blocks, each = Linear → BN → GELU → Dropout
#   Output: 10 logits  →  softmax (implicit in cross-entropy loss)
#
#   Why GELU?  Smoother gradients than ReLU; used in GPT, BERT etc.
#   Why BN?    Stabilises gradient magnitudes across layers (key for backprop)
#   Why skip?  Gives gradients a "highway" — reduces vanishing-gradient risk
# ─────────────────────────────────────────────────────────────────────────────

class ResidualBlock(nn.Module):
    """One hidden block with an optional residual (skip) connection."""
    def __init__(self, in_dim: int, out_dim: int, dropout: float = 0.3):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)
        self.bn     = nn.BatchNorm1d(out_dim)
        self.act    = nn.GELU()
        self.drop   = nn.Dropout(dropout)
        # 1×1 projection when dims differ  (needed for residual)
        self.proj   = nn.Linear(in_dim, out_dim, bias=False) if in_dim != out_dim else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.act(self.bn(self.linear(x))) + self.proj(x)   # ← skip connection
        # NOTE: the + is crucial for backprop — gradients flow through BOTH paths


class DeepMLP(nn.Module):
    def __init__(self, input_dim=3072, hidden_dims=(1024, 512, 256, 128),
                 num_classes=10, dropout=0.3):
        super().__init__()
        dims  = [input_dim] + list(hidden_dims)
        blocks= [ResidualBlock(dims[i], dims[i+1], dropout)
                 for i in range(len(dims)-1)]
        self.encoder = nn.Sequential(*blocks)
        self.head    = nn.Linear(dims[-1], num_classes)
        self._init_weights()

    def _init_weights(self):
        """Kaiming (He) init — keeps variance stable through deep GELU nets."""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.view(x.size(0), -1)   # flatten 32×32×3 → 3072
        z = self.encoder(x)
        return self.head(z)           # raw logits (no softmax — handled by loss)


model = DeepMLP().to(DEVICE)
print(f"\n[MODEL]\n{model}\n")
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"[INFO] Trainable parameters: {total_params:,}")



[MODEL]
DeepMLP(
  (encoder): Sequential(
    (0): ResidualBlock(
      (linear): Linear(in_features=3072, out_features=1024, bias=True)
      (bn): BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act): GELU(approximate='none')
      (drop): Dropout(p=0.3, inplace=False)
      (proj): Linear(in_features=3072, out_features=1024, bias=False)
    )
    (1): ResidualBlock(
      (linear): Linear(in_features=1024, out_features=512, bias=True)
      (bn): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act): GELU(approximate='none')
      (drop): Dropout(p=0.3, inplace=False)
      (proj): Linear(in_features=1024, out_features=512, bias=False)
    )
    (2): ResidualBlock(
      (linear): Linear(in_features=512, out_features=256, bias=True)
      (bn): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act): GELU(approximate='none')
      (drop): Dropout(p=0.3, inplace=False

In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# 3.  LOSS & OPTIMISER
#
#  CrossEntropyLoss = LogSoftmax + NLLLoss
#  ∂L/∂logit_i = softmax_i − 1[i == true_class]   (elegant closed form!)
#
#  SGD + Momentum:  v_t = β·v_{t-1} + ∇L
#                   W   = W − α·v_t
#
#  CosineAnnealingLR: smoothly decays lr → 0 over T_max epochs
# ─────────────────────────────────────────────────────────────────────────────
EPOCHS   = 30
LR       = 0.05
MOMENTUM = 0.9
WD       = 1e-4   # L2 weight-decay adds  λ·W  to gradients

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.SGD(model.parameters(), lr=LR, momentum=MOMENTUM,
                      weight_decay=WD, nesterov=True)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

In [12]:
# ─────────────────────────────────────────────────────────────────────────────
# 4.  TENSORBOARD SETUP
# ─────────────────────────────────────────────────────────────────────────────
writer = SummaryWriter(log_dir="runs/cifar10_backprop_demo")

# Log the model graph with a dummy input
dummy = torch.zeros(2, 3, 32, 32).to(DEVICE)
writer.add_graph(model, dummy)

# ─────────────────────────────────────────────────────────────────────────────
# 5.  GRADIENT HOOKS  –  capture per-layer gradient norms DURING backprop
#
#  register_full_backward_hook lets us intercept ∂L/∂output for each module
#  This is what makes backprop *visible* rather than a black box.
# ─────────────────────────────────────────────────────────────────────────────
gradient_norms: dict[str, float] = {}

def make_hook(name: str):
    def hook(module, grad_input, grad_output):
        # grad_output[0] is ∂L/∂(layer output)
        if grad_output[0] is not None:
            gradient_norms[name] = grad_output[0].norm().item()
    return hook

for name, module in model.named_modules():
    if isinstance(module, nn.Linear):
        module.register_full_backward_hook(make_hook(name))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.  TRAINING LOOP  –  forward → loss → backward → update
# ─────────────────────────────────────────────────────────────────────────────

def log_weight_histograms(epoch: int):
    """Push weight + gradient histograms to TensorBoard."""
    for tag, param in model.named_parameters():
        if param.requires_grad:
            writer.add_histogram(f"Weights/{tag}", param.data, epoch)
            if param.grad is not None:
                writer.add_histogram(f"Gradients/{tag}", param.grad, epoch)

def evaluate() -> tuple[float, float]:
    model.eval()
    correct = loss_sum = n = 0
    with torch.no_grad():
        for imgs, labels in test_ld:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            logits = model(imgs)
            loss_sum += criterion(logits, labels).item() * imgs.size(0)
            correct  += (logits.argmax(1) == labels).sum().item()
            n        += imgs.size(0)
    return loss_sum / n, correct / n * 100

global_step = 0

print("\n" + "═"*70)
print("  EPOCH │ TRAIN LOSS │ TRAIN ACC │  VAL LOSS │  VAL ACC │   LR")
print("═"*70)

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = train_correct = train_n = 0
    t0 = time.time()

    for step, (imgs, labels) in enumerate(train_ld):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

        # ── FORWARD PASS ──────────────────────────────────────────────────
        logits = model(imgs)              # shape: [B, 10]
        loss   = criterion(logits, labels)

        # ── BACKWARD PASS  (chain rule through the entire graph) ──────────
        optimizer.zero_grad()             # wipe stale gradients
        loss.backward()                   # ∂L/∂W computed for every parameter

        # Gradient clipping — prevents exploding gradients
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)

        # ── WEIGHT UPDATE ─────────────────────────────────────────────────
        optimizer.step()

        # ── LOGGING ───────────────────────────────────────────────────────
        bs = imgs.size(0)
        train_loss    += loss.item() * bs
        train_correct += (logits.argmax(1) == labels).sum().item()
        train_n       += bs

        if step % 30 == 0:
            # Log per-step loss + gradient norms to TensorBoard
            writer.add_scalar("Step/train_loss", loss.item(), global_step)
            for lname, gnorm in gradient_norms.items():
                writer.add_scalar(f"GradNorm/{lname}", gnorm, global_step)

        global_step += 1

    # ── EPOCH-LEVEL METRICS ───────────────────────────────────────────────
    tr_loss = train_loss / train_n
    tr_acc  = train_correct / train_n * 100
    val_loss, val_acc = evaluate()
    current_lr = scheduler.get_last_lr()[0]
    scheduler.step()

    # TensorBoard scalars
    writer.add_scalars("Loss", {"train": tr_loss, "val": val_loss}, epoch)
    writer.add_scalars("Accuracy", {"train": tr_acc, "val": val_acc}, epoch)
    writer.add_scalar("LearningRate", current_lr, epoch)

    # Weight + gradient histograms every 5 epochs
    if epoch % 5 == 0:
        log_weight_histograms(epoch)

    dt = time.time() - t0
    print(f"  {epoch:>5} │ {tr_loss:>10.4f} │ {tr_acc:>8.2f}% │"
          f" {val_loss:>9.4f} │ {val_acc:>7.2f}% │ {current_lr:.5f}  ({dt:.1f}s)")

print("═"*70)
print(f"\n[DONE] Training complete. Launch TensorBoard with:")
print(f"       tensorboard --logdir runs/\n")


══════════════════════════════════════════════════════════════════════
  EPOCH │ TRAIN LOSS │ TRAIN ACC │  VAL LOSS │  VAL ACC │   LR
══════════════════════════════════════════════════════════════════════


/tmp/ipython-input-207/4073716688.py:45: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.
  loss.backward()                   # ∂L/∂W computed for every parameter


      1 │     6.4552 │    17.93% │    3.2340 │   20.12% │ 0.05000  (61.2s)
      2 │     2.4100 │    26.08% │    1.9511 │   35.26% │ 0.04986  (61.0s)
      3 │     1.9103 │    35.08% │    1.8720 │   37.50% │ 0.04945  (65.6s)
      4 │     1.8531 │    38.02% │    1.7726 │   41.96% │ 0.04878  (61.4s)
      5 │     1.8168 │    39.75% │    1.8303 │   41.04% │ 0.04784  (62.9s)


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 7.  DEEP DIVE  –  Manually verify one backprop step
#
#  We freeze the batch, do a forward pass, print analytical gradients for
#  the final linear layer, then verify with PyTorch autograd.
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "═"*70)
print("  MANUAL BACKPROP VERIFICATION  (last linear layer)")
print("═"*70)

model.eval()
sample_imgs, sample_labels = next(iter(test_ld))
sample_imgs  = sample_imgs[:4].to(DEVICE)
sample_labels= sample_labels[:4].to(DEVICE)

# Forward
with torch.enable_grad():
    model.zero_grad()
    logits = model(sample_imgs)
    loss   = criterion(logits, sample_labels)
    loss.backward()

# Autograd gradient on the head weight
autograd_grad = model.head.weight.grad.clone()

# Analytical gradient for CrossEntropy + last Linear:
#   ∂L/∂W_head = (softmax(logits) − one_hot(y))^T @ z
with torch.no_grad():
    probs    = torch.softmax(logits.detach(), dim=1)
    one_hot  = torch.zeros_like(probs).scatter_(1, sample_labels.unsqueeze(1), 1)
    delta    = (probs - one_hot) / sample_imgs.size(0)   # [B,10]

    # Get the penultimate activations z (encoder output)
    z = model.encoder(sample_imgs.view(sample_imgs.size(0), -1))  # [B, 128]
    analytical_grad = delta.T @ z          # [10, 128]

err = (autograd_grad - analytical_grad).abs().max().item()
print(f"  Max absolute difference (autograd vs analytical): {err:.2e}")
print(f"  ✓ Verified: PyTorch autograd matches manual chain rule" if err < 1e-4
      else f"  ✗ Mismatch detected (err={err:.2e}) – check implementation")
print("═"*70)

writer.close()

# ─────────────────────────────────────────────────────────────────────────────
# 8.  WHAT TO LOOK FOR IN TENSORBOARD
# ─────────────────────────────────────────────────────────────────────────────
TENSORBOARD_GUIDE = """
TENSORBOARD PANELS & WHAT THEY REVEAL
══════════════════════════════════════

SCALARS → Loss
  • train & val curves converge  → model is learning
  • val >> train                 → overfitting (increase dropout / WD)
  • both plateau early           → lr too low or model capacity too small

SCALARS → Accuracy
  • should climb monotonically (with noise)
  • large train-val gap          → generalisation problem

SCALARS → LearningRate
  • cosine curve from 0.05 → 0   → you'll see how lr decay helps convergence

SCALARS → GradNorm / <layer>
  ★ THE MOST IMPORTANT PANEL FOR UNDERSTANDING BACKPROP ★
  • Early layers (encoder.0.linear) get gradients LAST (they're deepest)
  • Late layers  (head)            get gradients FIRST
  • If early-layer norms ≈ 0      → vanishing gradient (bad!)
  • If any norm >> 100            → exploding gradient (bad!)
  • Healthy range: 0.01 – 10.0

HISTOGRAMS → Weights/<layer>
  • Should spread out over time (network is learning diverse features)
  • If they collapse to 0         → dead neurons

HISTOGRAMS → Gradients/<layer>
  • Should be roughly centred on 0
  • Heavy tails indicate large updates; too-tight = network not moving

GRAPHS
  • Click to explore the full computation graph that .backward() traverses
"""
print(TENSORBOARD_GUIDE)